# GDRN D3 — Kaggle CPU preflight

Mục tiêu của notebook này là kiểm tra khả năng tái lập trước khi đo: Python/Java/k6/Docker, kết nối tới một server đã được người chạy cấu hình và khả năng đọc kết quả k6. Đây là artifact generator-only/preflight; notebook không tự publish database, service hoặc tunnel.

Tiêu chí: mọi capability được đánh dấu `available`, `not configured` hoặc `unavailable` với bằng chứng; không gọi đây là benchmark Kaggle hoàn tất nếu chưa có server và DB benchmark riêng. Số đo local là baseline local.

In [ ]:
from __future__ import annotations

import json
import os
import platform
import shutil
import subprocess
import sys
import urllib.error
import urllib.request
from pathlib import Path

SEED = int(os.environ.get('DATA_SEED', '20261024'))
ROOT = Path.cwd()
SEED


## Capability preflight

The full Spring/PostGIS server is not assumed to run inside Kaggle. A configured `BASE_URL` may point to a separately managed benchmark server, but this notebook never creates a public tunnel and never stores credentials in output.

In [ ]:
def version(command: list[str]) -> str:
    try:
        result = subprocess.run(command, capture_output=True, text=True, timeout=10)
        text = (result.stdout or result.stderr).strip().splitlines()
        return text[0] if text else f'exit={result.returncode}'
    except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
        return type(exc).__name__

def capability(executable: str, command: list[str]) -> dict:
    path = shutil.which(executable)
    return {'available': bool(path), 'path': path, 'version': version(command) if path else None}

capabilities = {
    'python': {'available': True, 'version': platform.python_version()},
    'java': capability('java', ['java', '-version']),
    'k6': capability('k6', ['k6', 'version']),
    'docker': capability('docker', ['docker', 'version', '--format', '{{.Server.Version}}']),
}
if capabilities['docker']['available']:
    capabilities['docker']['daemon'] = version(['docker', 'info', '--format', '{{.ServerVersion}}'])
else:
    capabilities['docker']['daemon'] = 'not available'

base_url = os.environ.get('BASE_URL', '').rstrip('/')
if not base_url:
    capabilities['server'] = {'status': 'not configured'}
else:
    try:
        with urllib.request.urlopen(f'{base_url}/actuator/health', timeout=5) as response:
            capabilities['server'] = {'status': 'reachable', 'httpStatus': response.status}
    except (urllib.error.URLError, TimeoutError) as exc:
        capabilities['server'] = {'status': 'unreachable', 'errorType': type(exc).__name__}

print(json.dumps(capabilities, indent=2))


## Reproducible configuration

Only non-secret configuration is displayed. `BENCHMARK_EMAIL` and `BENCHMARK_PASSWORD` are intentionally represented as booleans. The k6 run must use the repository workload and a token lifetime that covers warm-up + test + ramp-down; D3 treats expired-token failures as setup/configuration errors, not domain bottlenecks.

In [ ]:
safe_configuration = {
    'baseUrlConfigured': bool(os.environ.get('BASE_URL')),
    'emailConfigured': bool(os.environ.get('BENCHMARK_EMAIL')),
    'passwordConfigured': bool(os.environ.get('BENCHMARK_PASSWORD')),
    'workload': os.environ.get('WORKLOAD', 'read-page'),
    'vus': int(os.environ.get('VUS', '10')),
    'duration': os.environ.get('DURATION', '30s'),
    'warmup': os.environ.get('WARMUP', '10s'),
    'rampDown': os.environ.get('RAMP_DOWN', '10s'),
    'datasetSize': int(os.environ.get('DATASET_SIZE', '1000')),
    'seed': SEED,
    'jwtLifetimeSeconds': int(os.environ.get('JWT_LIFETIME_SECONDS', '900')),
}
safe_configuration


## Generator-only and server-on-Kaggle modes

- **Generator-only:** clone or upload this repository, point `BASE_URL` to a separately managed benchmark app and run `benchmark/generate_reports.py` followed by k6. This is the expected fallback when Kaggle lacks Java/PostGIS/Docker.
- **Server-on-Kaggle:** only mark this available after Java, the application build, a PostGIS-capable database, network access and the required credentials have each been verified. A missing Docker daemon is not silently treated as success.
- Do not create a public service, tunnel or shared database from this notebook. Use a private, disposable benchmark environment and record its topology outside secrets.

In [ ]:
def k6_summary(path: str | Path) -> dict:
    path = Path(path)
    if not path.exists():
        return {'status': 'not run', 'path': str(path)}
    payload = json.loads(path.read_text(encoding='utf-8'))
    metrics = payload.get('metrics', {})
    def value(metric: str, key: str):
        metric_payload = metrics.get(metric, {})
        values = metric_payload.get('values', metric_payload)
        return values.get(key)
    return {
        'status': 'available',
        'path': str(path),
        'requestsPerSecond': value('http_reqs', 'rate'),
        'averageMs': value('http_req_duration', 'avg'),
        'p50Ms': value('http_req_duration', 'med'),
        'p95Ms': value('http_req_duration', 'p(95)'),
        'p99Ms': value('http_req_duration', 'p(99)'),
        'errorRate': value('http_req_failed', 'rate'),
    }

summary_path = os.environ.get('K6_SUMMARY_PATH', 'benchmark/results/latest/summary.json')
k6_summary(summary_path)


## Run decision

The following cell is intentionally a dry-run decision record. Set `RUN_LIVE_BENCHMARK=True` only when `BASE_URL`, a disposable benchmark DB, credentials and k6 are explicitly available. The default notebook execution therefore remains safe and reproducible on a Kaggle CPU with no Docker daemon.

In [ ]:
RUN_LIVE_BENCHMARK = False
run_decision = {
    'status': 'not run',
    'reason': 'Live server, disposable DB and credentials must be explicitly configured; local results remain baseline.',
    'capabilities': capabilities,
    'configuration': safe_configuration,
}
if RUN_LIVE_BENCHMARK:
    if not base_url or not safe_configuration['emailConfigured'] or not safe_configuration['passwordConfigured']:
        raise RuntimeError('Live run requires BASE_URL and benchmark credentials through the environment.')
    if not capabilities['k6']['available']:
        raise RuntimeError('Native k6 is not available; run the repository Docker runner outside Kaggle or install k6 explicitly.')
    run_decision['status'] = 'ready to invoke repository runner'

print(json.dumps(run_decision, indent=2, default=str))
